# Libraries

In [1]:
#!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install "lm_eval[hf]"
!pip -q install "lm_eval[math]" #for MATH-500
!pip -q install --upgrade transformers
!pip -q install langdetect --break-system-packages #for If-eval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 79.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 3.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2

In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
MODEL_NAME       = "Llama-3.2-3B-Instruct-SparseGPT-70"
MODEL_PRETRAINED = "EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT"
SUB_FOLDER = "Models/70"
SEED             = 42

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Evaluations

**Configurations**

In [8]:
# Task definition
TASKS = [
    # Math
     Task("math-500",     "minerva_math500",           "math",                num_fewshot=0, batch_size=8, max_gen_toks=2048),
     Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=6,  max_gen_toks=1024),

    # Reasoning
     Task("gpqa_diamond", "gpqa_diamond_cot_zeroshot", "reasoning",           batch_size=8,  max_gen_toks=2048),
     Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=8),

    # Instruction Following
     ##Task("ifeval",       "ifeval",                    "instruction_following",batch_size=6,  max_gen_toks=1024),

    # Perplexity
    ##Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Knowledge
     ##Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),
]


# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    f"subfolder={SUB_FOLDER}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    #"enable_thinking=False",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [9]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "WARNING",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)

    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[MATH] math-500


2026-04-08:11:37:25 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-08:11:37:32 INFO     [_cli.run:376] Selected Tasks: ['minerva_math500']
2026-04-08:11:37:34 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 2048} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Map: 100%|██████████| 500/500 [00:00<00:00, 9090.46 examples/s]
2026-04-08:11:38:28 WARNING  [evaluator:333] Overwriting default num_fewshot of minerva_math500 from 4 to 0
2026-04-08:11:38:28 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 500/500 [3:00:23<00:00, 21.65s/it]  
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/70', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 2048}), limit: None, num_fewshot: 0, batch_size: 8
|     Tasks     |Version|Filter|n-shot|  Metric   |   |Value|   |Stderr|
|---------------|------:|------|-----:|-----------|---|----:|---|-----:|
|minerva_math500|      3|none  |     0|exact_match|↑  |0.000|±  | 0.000|
|               |       |none  |     0|math_verify|↑  |0.002|±  | 0.002|

Done

[MATH] gsm8k


2026-04-08:14:39:24 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-08:14:39:29 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-04-08:14:39:31 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 341036.06 examples/s]
2026-04-08:14:39:43 WARNING  [evaluator:333] Overwriting default num_fewshot of gsm8k_cot from 8 to 8
2026-04-08:14:39:43 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 1319/1319 [7:09:40<00:00, 19.55s/it]  
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/70', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 8, batch_size: 6
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value|   |Stderr|
|---------|------:|----------------|-----:|-----------|---|----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     8|exact_match|↑  |    0|±  |     0|
|         |       |strict-match    |     8|exact_match|↑  |    0|±  |     0|

Done

[REASONING] gpqa_diamond


2026-04-08:21:49:50 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-08:21:49:55 INFO     [_cli.run:376] Selected Tasks: ['gpqa_diamond_cot_zeroshot']
2026-04-08:21:49:57 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 2048} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Map: 100%|██████████| 198/198 [00:00<00:00, 1421.32 examples/s]
2026-04-08:21:50:12 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests:   0%|          | 0/198 [00:00<?, ?it/s]2026-04-08:21:50:13 WARNING  [models.huggingface:914] Left truncation applied. Original sequence length was 2816, truncating to last 2048 tokens. Some content will be lost.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBO

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/70', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 2048}), limit: None, num_fewshot: 0, batch_size: 8
|          Tasks          |Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|-------------------------|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gpqa_diamond_cot_zeroshot|      1|flexible-extract|     0|exact_match|↑  |0.0303|±  |0.0122|
|                         |       |strict-match    |     0|exact_match|↑  |0.0000|±  |0.0000|

Done

[REASONING] arc_challenge


2026-04-08:23:03:46 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-08:23:03:51 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
Generating validation split: 100%|██████████| 299/299 [00:00<00:00, 112404.49 examples/s]
2026-04-08:23:04:12 WARNING  [evaluator:333] Overwriting default num_fewshot of arc_challenge from None to 0
2026-04-08:23:04:12 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running loglikelihood requests: 100%|██████████| 4687/4687 [02:00<00:00, 38.81it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/70', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 8
|    Tasks    |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-------------|------:|------|-----:|--------|---|-----:|---|-----:|
|arc_challenge|      1|none  |     0|acc     |↑  |0.2048|±  |0.0118|
|             |       |none  |     0|acc_norm|↑  |0.2449|±  |0.0126|

Done


# Save reports

In [10]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)